# v01 — Data Validation Snapshot

범위: `riot-scale-2600` 데이터 규모와 strict validation 결과.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'src' / 'team2_surrender').exists()
)
os.chdir(ROOT)
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from IPython.display import display, Markdown, Image
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from team2_surrender.modeling import FEATURE_COLUMNS, TARGET_COLUMN, GROUP_COLUMN, make_group_split

RUN_ID = 'riot-scale-2600'
DATA_PATH = ROOT / 'data' / 'processed' / 'riot' / f'{RUN_ID}_team_features.csv'
VALIDATION_PATH = ROOT / 'data' / 'processed' / 'riot' / f'{RUN_ID}_team_features_validation_strict.json'
METRICS_PATH = ROOT / 'outputs' / 'metrics' / f'{RUN_ID}_model_comparison.json'
FIGURE_DIR = ROOT / 'reports' / 'figures' / RUN_ID
MODEL_DIR = ROOT / 'models' / RUN_ID
UPLOAD_VERIFICATION_PATH = ROOT / 'outputs' / 'verification' / f'supabase_upload_{RUN_ID}.json'

REQUIRED_PATHS = [DATA_PATH, VALIDATION_PATH, METRICS_PATH, FIGURE_DIR, MODEL_DIR, UPLOAD_VERIFICATION_PATH]

for path in REQUIRED_PATHS:
    print(f'{path.relative_to(ROOT)} -> {path.exists()}')

missing_paths = [path for path in REQUIRED_PATHS if not path.exists()]
if missing_paths:
    missing_text = '\n'.join(str(path.relative_to(ROOT)) for path in missing_paths)
    raise FileNotFoundError(
        'Missing required analysis artifacts:\n'
        f'{missing_text}\n\n'
        'Run: python scripts/prepare_jupyter_analysis_artifacts.py'
    )

data/processed/riot/riot-scale-2600_team_features.csv -> True
data/processed/riot/riot-scale-2600_team_features_validation_strict.json -> True
outputs/metrics/riot-scale-2600_model_comparison.json -> True
reports/figures/riot-scale-2600 -> True
models/riot-scale-2600 -> True
outputs/verification/supabase_upload_riot-scale-2600.json -> True


## 1. 데이터 로드

In [2]:
df = pd.read_csv(DATA_PATH)
y = df[TARGET_COLUMN].astype(bool).astype(int)
print('rows:', len(df))
print('matches:', df[GROUP_COLUMN].nunique())
print('positive rows:', int(y.sum()))
print('positive rate:', round(float(y.mean()), 4))
display(df.head())

rows: 5050
matches: 2525
positive rows: 870
positive rate: 0.1723


,match_id,team_id,feature_version,team_surrendered,queue_id,game_version,game_duration_sec,collected_at,gold_diff_15,kill_diff_15,tower_diff_15,dragon_diff_15,rift_herald_diff_15,cs_diff_15,avg_level_diff_15,first_blood,first_tower,ward_placed_diff_15,ward_kill_diff_15
0,KR_8143703690,100,v1_15min,True,420,16.6.756.931,1382,2026-05-14T19:59:29+00:00,-270,-6,1,-1,0,8,-0.4,1,1,-7,2
1,KR_8143703690,200,v1_15min,False,420,16.6.756.931,1382,2026-05-14T19:59:29+00:00,270,6,-1,1,0,-8,0.4,-1,-1,7,-2
2,KR_8143357958,100,v1_15min,False,420,16.6.756.931,2232,2026-05-14T19:59:29+00:00,2064,3,0,-2,0,9,0.2,-1,-1,1,-2
3,KR_8143357958,200,v1_15min,False,420,16.6.756.931,2232,2026-05-14T19:59:29+00:00,-2064,-3,0,2,0,-9,-0.2,1,1,-1,2
4,KR_8143287893,100,v1_15min,False,420,16.6.756.931,1473,2026-05-14T19:59:29+00:00,-4374,-9,0,-2,0,-74,-0.6,-1,0,1,-6


## 2. Dataset summary

In [3]:
summary = pd.DataFrame({
    'metric': [
        'team_rows', 'unique_matches', 'positive_rows', 'negative_rows', 'positive_rate',
        'min_duration_sec', 'median_duration_sec', 'max_duration_sec', 'queue_ids'
    ],
    'value': [
        len(df),
        df[GROUP_COLUMN].nunique(),
        int(y.sum()),
        int((1 - y).sum()),
        round(float(y.mean()), 4),
        int(df['game_duration_sec'].min()),
        int(df['game_duration_sec'].median()),
        int(df['game_duration_sec'].max()),
        ', '.join(map(str, sorted(df['queue_id'].unique()))),
    ]
})
display(summary)

,metric,value
0,team_rows,5050
1,unique_matches,2525
2,positive_rows,870
3,negative_rows,4180
4,positive_rate,0.1723
5,min_duration_sec,910
6,median_duration_sec,1687
7,max_duration_sec,2962
8,queue_ids,420


## 3. Strict validation checks

In [4]:
validation = json.loads(VALIDATION_PATH.read_text())
checks = validation.get('checks', validation)
rows = []
if isinstance(checks, dict):
    for name, value in checks.items():
        if isinstance(value, dict):
            rows.append({'check': name, 'status': value.get('status') or value.get('result'), 'message': value.get('message') or value.get('detail') or ''})
        else:
            rows.append({'check': name, 'status': value, 'message': ''})
else:
    rows = checks
validation_df = pd.DataFrame(rows)
display(validation_df)

,evidence,message,name,ok
0,"[match_id, team_id, feature_version, team_surr...",All required columns are present,required_columns,True
1,5050,Row count is sufficient,row_count,True
2,"[gold_diff_15, kill_diff_15, tower_diff_15, dr...",At least 10 feature columns present,feature_count,True
3,None,No forbidden identifier/key columns found,privacy_columns,True
4,"{'positive_count': 870, 'values': ['true', 'fa...",Target contains at least two classes,target_classes,True
5,2525,At least 3 unique match groups present,group_count,True
6,None,Every match has exactly two team rows,two_rows_per_match,True
7,None,No match has more than one surrender-defeat team,positive_labels_per_match,True
8,None,All rows use queue_id=420,queue_id,True
9,None,All rows have game_duration_sec >= 900,duration,True


## 4. Upload row count check

In [5]:
upload = json.loads(UPLOAD_VERIFICATION_PATH.read_text())
display(pd.DataFrame([upload]))

,status,mode,verified_at,input_csv,expected_matches,expected_team_rows,expected_positive_rows,actual_matches,actual_team_rows,actual_positive_rows,note
0,complete,local_csv_dry_run,2026-05-26T06:57:40+00:00,data/processed/riot/riot-scale-2600_team_featu...,2525,5050,870,2525,5050,870,Generated from local CSV for notebook reproduc...


## v01 notes

- 데이터셋 규모: 2,525 matches / 5,050 team rows.
- match별 2개 team row, queue 420, 15분 이상 경기 조건, positive label 제약 통과.